In [ ]:
!pip install -q kaggle
!pip install -q nba_api

In [ ]:
#ANN final project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import os
import time
from datetime import datetime
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error

In [ ]:
# KAGGLE API SETUP

# Configure Kaggle credentials securely in your environment before running.
# Never paste API keys into a notebook or commit them to GitHub.

import os
assert os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"), "Set KAGGLE_USERNAME and KAGGLE_KEY first"

!kaggle datasets download -d mexwell/nba-shots -p /content/nba_shots --unzip
print("SHOT DATA FILES:")
print(os.listdir("/content/nba_shots"))


In [ ]:
# LOAD ALL SHOT DATA

shots_path = "/content/nba_shots"

dataframes = []

for year in range(2004, 2025):
    filename = f"NBA_{year}_Shots.csv"
    full_path = os.path.join(shots_path, filename)

    try:
        yearly_df = pd.read_csv(full_path)
        dataframes.append(yearly_df)
        print(f"Loaded: {filename}")

    except FileNotFoundError:
        print(f"Missing: {filename}")

# Merge all seasons together
shots_df = pd.concat(dataframes, ignore_index=True)

print("\nMerge complete")
print(f"Rows: {len(shots_df):,}")
print(f"Columns: {shots_df.shape[1]}")

In [ ]:

def create_wide_zone_stats_corrected(df):
    # 1. Filter for 3PT Field Goals only
    threes_df = df[df['SHOT_TYPE'] == '3PT Field Goal'].copy()

    # 2. Convert True/False to 1/0 for easy math
    threes_df['SHOT_MADE_INT'] = threes_df['SHOT_MADE'].astype(int)

    # 3. Create columns specifically for MAKE locations.
    # If it's a miss (False), set location to NaN so it's ignored in the average.
    threes_df['MAKE_LOC_X'] = np.where(threes_df['SHOT_MADE'] == True, threes_df['LOC_X'], np.nan)
    threes_df['MAKE_LOC_Y'] = np.where(threes_df['SHOT_MADE'] == True, threes_df['LOC_Y'], np.nan)

    # 4. Group by Player, Season (using SEASON_1 for the year integer), and Basic Zone
    zone_group = threes_df.groupby(['PLAYER_ID', 'PLAYER_NAME', 'SEASON_1', 'BASIC_ZONE'])

    # 5. Aggregate all your requested metrics
    zone_stats = zone_group.agg(
        FGA=('SHOT_MADE_INT', 'count'),
        FGM=('SHOT_MADE_INT', 'sum'),
        AVG_SHOT_X=('LOC_X', 'mean'),
        AVG_SHOT_Y=('LOC_Y', 'mean'),
        AVG_MAKE_X=('MAKE_LOC_X', 'mean'),
        AVG_MAKE_Y=('MAKE_LOC_Y', 'mean')
    ).reset_index()

    # 6. Calculate percentage per zone
    zone_stats['FG_PCT'] = zone_stats['FGM'] / zone_stats['FGA']

    # 7. PIVOT the data into the "Wide" format for the Neural Net
    wide_df = zone_stats.pivot_table(
        index=['PLAYER_ID', 'PLAYER_NAME', 'SEASON_1'],
        columns='BASIC_ZONE',
        values=['FGA', 'FGM', 'FG_PCT', 'AVG_SHOT_X', 'AVG_SHOT_Y', 'AVG_MAKE_X', 'AVG_MAKE_Y']
    )

    # 8. Flatten the multi-level column names (e.g., "Above the Break 3_FGA")
    wide_df.columns = [f"{zone}_{metric}" for metric, zone in wide_df.columns]
    wide_df = wide_df.reset_index()

    # 9. Fill missing Attempts (FGA) and Makes (FGM) with 0 for zones a player didn't shoot from
    fga_fgm_cols = [col for col in wide_df.columns if 'FGA' in col or 'FGM' in col]
    wide_df[fga_fgm_cols] = wide_df[fga_fgm_cols].fillna(0)

    return wide_df


In [ ]:
player_season_df = create_wide_zone_stats_corrected(shots_df)

print(player_season_df.shape)
player_season_df.head()

In [ ]:
# GET ONLY PLAYERS IN SHOT DATA

player_ids_needed = (
    shots_df['PLAYER_ID']
    .dropna()
    .unique()
)

print(f"Players needed: {len(player_ids_needed)}")

In [ ]:
from nba_api.stats.endpoints import commonplayerinfo

player_info_list = []

for i, player_id in enumerate(player_ids_needed):

    try:
        info = commonplayerinfo.CommonPlayerInfo(
            player_id=int(player_id)
        )

        temp_df = info.get_data_frames()[0]

        player_info_list.append(temp_df)

        if i % 25 == 0:
            print(f"{i}/{len(player_ids_needed)}")

        time.sleep(0.5)

    except Exception as e:
        print(f"Failed {player_id}: {e}")

player_bio_df = pd.concat(
    player_info_list,
    ignore_index=True
)

print(player_bio_df.shape)

In [ ]:
player_bio_df = player_bio_df[[
    'PERSON_ID',
    'DISPLAY_FIRST_LAST',
    'HEIGHT',
    'WEIGHT',
    'BIRTHDATE',
    'SEASON_EXP',
    'POSITION',
    'DRAFT_YEAR',
    'DRAFT_NUMBER'
]]

player_bio_df.to_csv(
    '/content/player_bio_df.csv',
    index=False
)

In [ ]:
player_bio_df = pd.read_csv(
    '/content/player_bio_df.csv'
)

In [ ]:
def height_to_inches(height_str):
    try:
        feet, inches = height_str.split("-")
        return int(feet) * 12 + int(inches)
    except:
        return None


player_bio_df["height_inches"] = player_bio_df["HEIGHT"].apply(height_to_inches)

player_bio_df["weight_lbs"] = pd.to_numeric(
    player_bio_df["WEIGHT"],
    errors="coerce"
)

player_bio_df["birthdate"] = pd.to_datetime(
    player_bio_df["BIRTHDATE"],
    errors="coerce"
)

player_bio_df["season_exp"] = pd.to_numeric(
    player_bio_df["SEASON_EXP"],
    errors="coerce"
)

player_bio_df["draft_number"] = pd.to_numeric(
    player_bio_df["DRAFT_NUMBER"],
    errors="coerce"
)

player_bio_df["undrafted"] = player_bio_df["draft_number"].isna().astype(int)

player_bio_clean_df = player_bio_df[[
    "PERSON_ID",
    "DISPLAY_FIRST_LAST",
    "height_inches",
    "weight_lbs",
    "birthdate",
    "season_exp",
    "POSITION",
    "draft_number",
    "undrafted"
]].copy()

player_bio_clean_df.head()

In [ ]:
model_df = player_season_df.merge(
    player_bio_clean_df,
    left_on="PLAYER_ID",
    right_on="PERSON_ID",
    how="left"
)

print(model_df.shape)

In [ ]:
model_df["season_year"] = (
    model_df["SEASON_1"] #current season
    .astype(str)
    .str[:4]
    .astype(int)
)

model_df["season_start_date"] = pd.to_datetime(
    model_df["season_year"].astype(str) + "-10-01"
)

model_df["age"] = (
    (model_df["season_start_date"] - model_df["birthdate"])
    .dt.days / 365.25
)

In [ ]:
model_df = model_df.drop(
    columns=[
        "PERSON_ID",
        "DISPLAY_FIRST_LAST",
        "birthdate",
        "season_start_date"
    ],
    errors="ignore"
)

model_df.head()

In [ ]:
model_df.to_csv("/content/model_df.csv", index=False)

In [ ]:
model_df = pd.read_csv("/content/model_df.csv")

In [ ]:
model_df.columns = (
    model_df.columns
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('-', '_')
)

In [ ]:
# CREATE CURRENT 3PT STATS

three_point_zones = [
    'above_the_break_3',
    'left_corner_3',
    'right_corner_3'
]

model_df['total_3pa'] = sum(
    model_df[f'{zone}_fga']
    for zone in three_point_zones
)

model_df['total_3pm'] = sum(
    model_df[f'{zone}_fgm']
    for zone in three_point_zones
)

model_df['current_3p_pct'] = (
    model_df['total_3pm']
    / model_df['total_3pa']
)

In [ ]:
# COMBINE LEFT + RIGHT CORNER 3s

model_df['corner_3_fga'] = (
    model_df['left_corner_3_fga']
    + model_df['right_corner_3_fga']
)

model_df['corner_3_fgm'] = (
    model_df['left_corner_3_fgm']
    + model_df['right_corner_3_fgm']
)

model_df['corner_3_fg_pct'] = (
    model_df['corner_3_fgm']
    / model_df['corner_3_fga']
)

# Average depth/location
model_df['corner_3_avg_shot_y'] = (
    (
        model_df['left_corner_3_avg_shot_y']
        * model_df['left_corner_3_fga']
    )
    +
    (
        model_df['right_corner_3_avg_shot_y']
        * model_df['right_corner_3_fga']
    )
) / model_df['corner_3_fga']

In [ ]:
# ==========================================
# CREATE DELTA 3P TARGET
# ==========================================

model_df = model_df.sort_values(
    ['player_id', 'season_year']
).copy()

model_df['next_3p_pct'] = (
    model_df.groupby('player_id')['current_3p_pct']
    .shift(-1)
)

model_df['next_season_year'] = (
    model_df.groupby('player_id')['season_year']
    .shift(-1)
)

model_df['is_true_next_season'] = (
    model_df['next_season_year'] == model_df['season_year'] + 1
)

training_df = model_df[
    (model_df['is_true_next_season']) &
    (model_df['total_3pa'] >= 75)
].copy()

training_df['delta_3p'] = (
    training_df['next_3p_pct'] - training_df['current_3p_pct']
)

print(training_df.shape)
training_df[['player_name', 'season_year', 'current_3p_pct', 'next_3p_pct', 'delta_3p']].head()

In [ ]:
# SMOOTH LOW-VOLUME CORNER 3P%

MIN_CORNER_ATTEMPTS = 20

league_corner_pct = training_df['corner_3_fg_pct'].mean()

training_df['corner_3_fg_pct_adj'] = np.where(
    training_df['corner_3_fga'] >= MIN_CORNER_ATTEMPTS,
    training_df['corner_3_fg_pct'],
    league_corner_pct
)

In [ ]:
training_df['corner_share'] = (
    training_df['corner_3_fga']
    / training_df['total_3pa']
)

training_df['above_break_share'] = (
    training_df['above_the_break_3_fga']
    / training_df['total_3pa']
)

In [ ]:
shares = np.vstack([
    training_df['corner_share'],
    training_df['above_break_share']
]).T

training_df['shot_entropy'] = -(
    shares * np.log(shares + 1e-10)
).sum(axis=1)

In [ ]:
training_df['young_player'] = (
    training_df['age'] <= 25
).astype(int)

training_df['prime_player'] = (
    (training_df['age'] > 25) &
    (training_df['age'] <= 31)
).astype(int)

training_df['aging_player'] = (
    training_df['age'] > 31
).astype(int)

In [ ]:
training_df = training_df.sort_values(
    ['player_id', 'season_year']
)

training_df['prev_3p_pct'] = (
    training_df.groupby('player_id')
    ['current_3p_pct']
    .shift(1)
)

training_df['last_year_delta'] = (
    training_df['current_3p_pct']
    - training_df['prev_3p_pct']
)

In [ ]:
LEAGUE_3P = training_df['current_3p_pct'].mean()
K = 100

training_df['bayesian_3p_pct'] = (
    (
        training_df['total_3pm']
        + K * LEAGUE_3P
    )
    /
    (
        training_df['total_3pa']
        + K
    )
)

In [ ]:
training_df['zone_balance'] = abs(
    training_df['corner_share']
    -
    training_df['above_break_share']
)

In [ ]:
training_df['corner_vs_above_gap'] = (
    training_df['corner_3_fg_pct_adj']
    -
    training_df['above_the_break_3_fg_pct']
)

In [ ]:
feature_columns = [

    'bayesian_3p_pct',
    'total_3pa',

    'corner_share',
    'above_break_share',

    'corner_3_fg_pct_adj',
    'above_the_break_3_fg_pct',

    'above_the_break_3_avg_shot_y',

    'age',
    'season_exp',

    'young_player',
    'prime_player',
    'aging_player',

    'undrafted'
]

In [ ]:
training_df = training_df.dropna(
    subset=feature_columns + ['delta_3p']
).copy()

X = training_df[feature_columns]
y = training_df['delta_3p']

print(X.shape)
print(y.describe())

In [ ]:
groups = training_df['player_id']

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [ ]:
#As a benchmark, we compare our prediction to just predicting the same percentage.
from sklearn.metrics import mean_absolute_error, mean_squared_error

baseline_pred = np.zeros(len(y_test))

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()

linear_model.fit(X_train_scaled, y_train)

linear_pred = linear_model.predict(X_test_scaled)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))

print("Linear Regression MAE:", linear_mae)
print("Linear Regression RMSE:", linear_rmse)

In [ ]:
#We are training a small model because we have a relatively small dataset

ann_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.20),

    Dense(32, activation='relu'),
    Dropout(0.10),

    Dense(16, activation='relu'),

    Dense(1)
])

ann_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [ ]:
history = ann_model.fit(
    X_train_scaled,
    y_train,

    validation_split=0.20,

    epochs=100,
    batch_size=32,

    callbacks=[early_stop],

    verbose=1
)

In [ ]:
ann_pred = ann_model.predict(X_test_scaled).flatten()

ann_mae = mean_absolute_error(y_test, ann_pred)
ann_rmse = np.sqrt(mean_squared_error(y_test, ann_pred))

print("Baseline MAE:", baseline_mae)
print("Linear Regression MAE:", linear_mae)
print("ANN MAE:", ann_mae)

print()

print("Baseline RMSE:", baseline_rmse)
print("Linear Regression RMSE:", linear_rmse)
print("ANN RMSE:", ann_rmse)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 6))

plt.scatter(
    y_test,
    ann_pred,
    alpha=0.35
)

plt.axhline(0, linestyle='--')
plt.axvline(0, linestyle='--')

plt.xlabel("Actual Change in 3P%")
plt.ylabel("Predicted Change in 3P%")
plt.title("ANN Predictions of Year-to-Year 3P% Change")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('ANN Training and Validation Loss')
plt.legend()

plt.show()

In [ ]:
results_summary = pd.DataFrame({
    'Model': ['Baseline: No Change', 'Linear Regression', 'ANN'],
    'MAE': [baseline_mae, linear_mae, ann_mae],
    'RMSE': [baseline_rmse, linear_rmse, ann_rmse]
})

results_summary